# Ladybird — Build en Colab
Compila **Ladybird** (el navegador) solo para las plataformas que actives.

**Cómo usar:**
1. *(Opcional)* Montá Google Drive para cachear entre sesiones:
   ```
   from google.colab import drive; drive.mount('/content/drive')
   ```
2. Editá la celda **CONFIGURACIÓN** y poné `True` en lo que quieras compilar.
3. Ejecutá las celdas en orden (menú → *Entorno de ejecución → Ejecutar todo*).
   Cada celda compila **solo si su variable está en True**.

**Qué hace cada opción:**
| Opción | Genera | Corré en |
|---|---|---|
| `COMPILE_LINUX` | binario Linux (Qt) | WSL2/WSLg de Windows, Linux, X11 |
| `COMPILE_ANDROID` | APK `arm64-v8a` | tu teléfono |
| `COMPILE_WINDOWS` | — | ⛔ no se puede desde Linux (ver celda final) |

> ⚠️ Cada build es enorme (varias horas) y usa casi toda la RAM de Colab.


In [ ]:
# ===================== CONFIGURACIÓN =====================
# Poné True en las plataformas que querés compilar y ejecutá todo.

COMPILE_LINUX   = True    # Linux (Qt) -> corre en WSL2/WSLg de Windows
COMPILE_ANDROID = True    # Android APK
COMPILE_WINDOWS = False   # Windows nativo: NO se puede desde Linux

# ---------- opciones ----------
REPO_URL    = "https://github.com/eduardo-bertey/ladybird.git"
DRIVE_ROOT  = "/content/drive/MyDrive/ladybird-build"   # caches persistentes
LINUX_JOBS  = 4               # paralelismo Linux (Colab ~12 GB RAM)
ANDROID_ABI = "arm64-v8a"     # arm64-v8a = teléfono | x86_64 = emulador
ANDROID_JOBS = 4
QT_VERSION  = "6.9.3"         # Qt >= 6.9 para el build Linux


In [ ]:
import os, subprocess, textwrap

def bash(script, verbose=True):
    script = textwrap.dedent(script)
    if verbose:
        print("$ bash -e" )
        print(script)
    env = dict(os.environ)
    for k, v in globals().items():
        if isinstance(v, (str, int, float, bool)):
            env[k] = str(v)
    r = subprocess.run(["/bin/bash", "-e", "-c", script], text=True, capture_output=True, env=env)
    if r.returncode != 0:
        out = (r.stdout or "").strip()
        err = (r.stderr or "").strip()
        print("=== STDOUT (ultimas 40 lineas) ===")
        print("\n".join(out.splitlines()[-40:]) if out else "(vacio)")
        print("=== STDERR (ultimas 40 lineas) ===")
        print("\n".join(err.splitlines()[-40:]) if err else "(vacio)")
        raise RuntimeError(f"FALLO (exit {r.returncode})")
    if verbose and r.stdout:
        print(r.stdout)
    return r

# Si no montaste Drive, usá una carpeta local
if not os.path.isdir("/content/drive/MyDrive"):
    DRIVE_ROOT = "/content/ladybird-build"
print("DRIVE_ROOT:", DRIVE_ROOT)
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(os.path.expanduser("~/.local/bin"), exist_ok=True)

# ----- dependencias base (apt) -----
bash(r"""
sudo apt-get update -qq
sudo apt-get install -y -qq --no-install-recommends \
  curl unzip zip git ccache python3-pip python3-venv pkg-config file nasm ninja-build \
  autoconf autoconf-archive automake libtool libncurses-dev libdrm-dev libgl1-mesa-dev \
  libxkbcommon-dev libx11-dev libxcb1-dev libpulse-dev glslang-tools fonts-liberation2 \
  software-properties-common
""")

# ----- cmake >= 3.30 (pip trae la última; Colab trae vieja) -----
bash(r"""
pip3 install --quiet --user -U cmake pyyaml requests six
export PATH="$HOME/.local/bin:$PATH"
cmake --version | head -1
""")

# ----- Rust 1.96.1 (fijado por rust-toolchain.toml) -----
bash(r"""
if ! command -v rustup >/dev/null 2>&1; then
  curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal
fi
source "$HOME/.cargo/env"
rustup toolchain install 1.96.1 --profile minimal
""")


In [ ]:
if COMPILE_LINUX:
    bash(r"""
set -euo pipefail
export PATH="$HOME/.local/bin:$PATH"
source "$HOME/.cargo/env"

# --- gcc-14 (Ubuntu 24.04 trae gcc-13; ladybird pide >= 14) ---
if ! command -v gcc-14 >/dev/null 2>&1; then
  sudo add-apt-repository -y ppa:ubuntu-toolchain-r/test >/dev/null 2>&1 || true
  sudo apt-get update -qq
  sudo apt-get install -y -qq g++-14
fi
sudo update-alternatives --install /usr/bin/gcc gcc /usr/bin/gcc-14 100 \
    --slave /usr/bin/g++ g++ /usr/bin/g++-14 2>/dev/null || true
gcc --version | head -1

# --- Qt >= 6.9 vía aqtinstall (Ubuntu trae Qt 6.4, muy vieja) ---
pip3 install --quiet aqtinstall
if [ ! -d "$HOME/Qt" ]; then
  for V in "$QT_VERSION" 6.9.3 6.9.2 6.9.1 6.9.0; do
    if aqt install-qt linux desktop "$V" gcc_64 -O "$HOME/Qt"; then
      break
    fi
  done
fi
QT_PREFIX=$(ls -d $HOME/Qt/*/gcc_64 | head -1)
echo "Qt: $QT_PREFIX"

# --- obtener el repo (clonar si no existe, actualizar si hay cambios) ---
if [ ! -d "$HOME/ladybird/.git" ]; then
  git clone --depth 1 "$REPO_URL" "$HOME/ladybird"
fi
cd "$HOME/ladybird"
BEFORE=$(git rev-parse --short HEAD)
git fetch origin master > /dev/null 2>&1
git reset --hard origin/master > /dev/null 2>&1
AFTER=$(git rev-parse --short HEAD)
if [ "$BEFORE" != "$AFTER" ]; then
  echo "Repo actualizado: $BEFORE -> $AFTER"
else
  echo "Repo al dia ($AFTER)"
fi

# --- configurar y compilar ---
cmake --preset Release -B Build \
  -DCMAKE_PREFIX_PATH="$QT_PREFIX" \
  -DCMAKE_BUILD_PARALLEL_LEVEL="$LINUX_JOBS"
cmake --build Build --parallel "$LINUX_JOBS"

echo "=========================================="
echo "LISTO -> Build/bin/Ladybird"
echo "En Windows: copialo a tu WSL2 y ejecutalo (corre con WSLg)."
echo "=========================================="
""")
else:
    print("LINUX desactivado (COMPILE_LINUX = False)")


In [ ]:
if COMPILE_ANDROID:
    bash(r"""
set -euo pipefail
export PATH="$HOME/.local/bin:$PATH"
export ANDROID_HOME="${ANDROID_HOME:-$HOME/android-sdk}"

# --- dependencias base (apt) ---
sudo apt-get update -qq > /dev/null 2>&1 || true
sudo apt-get install -y -qq openjdk-17-jdk-headless cmake ninja-build zip unzip curl git ccache > /dev/null 2>&1 || true

# --- JDK 17 (AGP 8.11 lo requiere; Colab trae Java 21 por defecto) ---
if [ ! -d /usr/lib/jvm/java-17-openjdk-amd64 ]; then
  sudo apt-get install -y -qq openjdk-17-jdk-headless > /dev/null 2>&1 || true
fi
JAVA17=$(ls -d /usr/lib/jvm/java-17-* 2>/dev/null | head -1)
export JAVA_HOME="$JAVA17"
export PATH="$JAVA_HOME/bin:$PATH"
java -version 2>&1 | head -1
export PATH="$JAVA_HOME/bin:$PATH"
java -version 2>&1 | head -1

# --- Rust 1.96.1 (necesario para los crates del build) ---
if ! command -v cargo > /dev/null 2>&1; then
  curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain 1.96.1 > /dev/null 2>&1
fi
export PATH="$HOME/.cargo/bin:$PATH"
rustc --version

# --- Android SDK + NDK 29 + CMake (cache en Drive) ---
if [ ! -d "$ANDROID_HOME/cmdline-tools/latest" ]; then
  SDK_ZIP="$DRIVE_ROOT/commandlinetools.zip"
  [ -f "$SDK_ZIP" ] || curl -o "$SDK_ZIP" "https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip"
  unzip -q -o "$SDK_ZIP" -d /tmp/cmdline-tools
  mkdir -p "$ANDROID_HOME/cmdline-tools/latest"
  mv /tmp/cmdline-tools/cmdline-tools/* "$ANDROID_HOME/cmdline-tools/latest/"
fi
export PATH="$ANDROID_HOME/cmdline-tools/latest/bin:$PATH"
yes | sdkmanager --licenses > /dev/null 2>&1 || true
sdkmanager "platform-tools" "build-tools;35.0.0" "platforms;android-35" "ndk;29.0.13599879" "cmake;3.31.5" 2>&1 | tail -1

# --- obtener el repo (clonar si no existe, actualizar si hay cambios) ---
if [ ! -d "$HOME/ladybird/.git" ]; then
  git clone --depth 1 "$REPO_URL" "$HOME/ladybird"
fi
cd "$HOME/ladybird"
BEFORE=$(git rev-parse --short HEAD)
git fetch origin master > /dev/null 2>&1
git reset --hard origin/master > /dev/null 2>&1
AFTER=$(git rev-parse --short HEAD)
if [ "$BEFORE" != "$AFTER" ]; then
  echo "Repo actualizado: $BEFORE -> $AFTER"
else
  echo "Repo al dia ($AFTER)"
fi

# --- vcpkg ---
python3 Meta/Utils/build_vcpkg.py

# --- caches persistentes ---
export CCACHE_DIR="$DRIVE_ROOT/ccache"
export XDG_CACHE_HOME="$DRIVE_ROOT/xdg-cache"
export GRADLE_OPTS="-Xmx2048m"

# --- build APK (tarda horas) ---
cd UI/Android
./gradlew --stop > /dev/null 2>&1 || true
NPROC=$(nproc); LIMIT=$(( NPROC < ANDROID_JOBS ? NPROC : ANDROID_JOBS ))
taskset -c 0-$((LIMIT-1)) ./gradlew assembleDebug \
  -Pandroid.injected.build.abi="$ANDROID_ABI" --console=plain

APK=$(ls build/outputs/apk/debug/app-debug.apk 2>/dev/null || true)
if [ -n "$APK" ]; then
  mkdir -p /content/apk
  cp "$APK" "/content/apk/ladybird-${ANDROID_ABI}.apk"
  echo "=========================================="
  echo "LISTO -> /content/apk/ladybird-${ANDROID_ABI}.apk"
  echo "=========================================="
else
  echo "No se encontró el APK (revisá los logs de gradle)"
fi
""")
else:
    print("ANDROID desactivado (COMPILE_ANDROID = False)")


In [ ]:
if COMPILE_WINDOWS:
    print(textwrap.dedent('''
    ⛔ Windows NATIVO (.exe) NO se puede compilar en Colab/Linux.

    Por qué:
     - Ladybird para Windows se compila con MSVC/ClangCL (toolchain de Microsoft,
       solo corre en Windows).
     - MinGW/MSYS2 NO está soportado (está en la doc oficial del repo).
     - No existe un linker MSVC que funcione en Linux.

    Opciones que SÍ funcionan:
     1) Compilá la opción LINUX de este cuaderno y corré el binario en WSL2 con WSLg
        (WSL2 es Linux real dentro de Windows -> ese binario corre directo, sin .exe).
     2) Build nativo Windows en un runner windows de GitHub Actions (necesita MSVC).
        Si querés, agrego ese job al workflow de tu fork.
    '''))
else:
    print("WINDOWS desactivado (COMPILE_WINDOWS = False)")
